# Learning a Burgers forcing-to-solution map

With `u0="random"` and `f="random"`, `Burgers.generate` manufactures exact pairs of a solution `u(t, x)` and the forcing `f(t, x)` that drives it, without integrating anything. SpectralSVR learns `f -> u` over the joint time and space coefficients.

In [ ]:
import torch

from SpectralSVR import LSSVR, Burgers, FourierBasis, SpectralSVR

generator = torch.Generator().manual_seed(0)
u, f = Burgers().generate(
    FourierBasis, n=300, modes=(8, 16), u0='random', f='random',
    generator=generator, time_dependent_coeff=False,
)
samples_of_f = f.get_values().real.flatten(1)
train, test = slice(0, 240), slice(240, 300)

## Fit on 240 pairs, score on the other 60

In [ ]:
model = SpectralSVR(FourierBasis(domain=u.domain), LSSVR(C=1e3))
model.train(samples_of_f[train], u.coeff[train])
model.test(samples_of_f[test], u.coeff[test])

For a solution integrated from an initial condition rather than manufactured, pass a `u0` and `f=0`; `Burgers.generate` then runs its ETDRK4 solver.